In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

batch_size = 128
epochs = 10
lr = 1e-3
mc_iters = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(0)
torch.manual_seed(0)



In [2]:
class CNN(nn.Module):
    def __init__(self, dropout_p=0.3, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=dropout_p)

        self.react_thresh = 3.5
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x, react=False):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv3(x)))
        x = self.dropout(x)

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        if react and self.react_thresh:
            x = torch.clamp(x, max=self.react_thresh)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [3]:
transform_cifar10 = transforms.Compose([
    transforms.ToTensor()
])

transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.expand(3, -1, -1))
])

train_id = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar10)
test_id = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar10)
test_ood = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

train_id_loader = DataLoader(train_id, batch_size=batch_size, shuffle=True)
test_id_loader = DataLoader(test_id, batch_size=batch_size, shuffle=False)
test_ood_loader = DataLoader(test_ood, batch_size=batch_size, shuffle=False)


100%|██████████| 170M/170M [00:05<00:00, 28.5MB/s] 
100%|██████████| 9.91M/9.91M [00:00<00:00, 11.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 377kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.20MB/s]


In [4]:
def train(model, train_loader, epochs, lr):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct_samples = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1)

            correct_samples += (preds == y).sum().item()

        accuracy_percent = correct_samples / len(train_loader) * 100
        print(f'Epoch {epoch}; Train loss {total_loss / len(train_loader)}; Accuracy {accuracy_percent}')


In [ ]:
def compute_ood_metrics(id_scores, ood_scores):
    y_true = np.concatenate([
        np.zeros_like(id_scores),
        np.ones_like(ood_scores)
    ])
    scores = np.concatenate([id_scores, ood_scores])

    fpr, tpr, _ = roc_curve(y_true, scores)
    target_tpr = 0.95
    idxs = np.where(tpr >= target_tpr)[0]
    if len(idxs) > 0:
        fpr95 = fpr[idxs[0]]
    else:
        fpr95 = 1.0

    auroc = roc_auc_score(y_true, scores)
    aupr = average_precision_score(y_true, scores)

    print(f'AUROC {auroc}')
    print(f'AUPR {aupr}')
    print(f'FPR@95%TPR {fpr95}')

    return auroc, aupr, fpr95


In [6]:
def get_softmax_ood_scores(model, id_loader, ood_loader):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            top_probs, _ = probs.max(dim=1)
            scores = 1.0 - top_probs

            id_scores.append(scores.cpu().numpy())

    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            top_probs, _ = probs.max(dim=1)
            scores = 1.0 - top_probs

            ood_scores.append(scores.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores



def mcd_entropy(model, x, T=20):
    model.to(device)
    model.train()

    with torch.no_grad():
        probs_T = []
        for _ in range(T):
            logits = model(x)
            probs = F.softmax(logits, dim=1)

            probs_T.append(probs.unsqueeze(0))

        probs_T = torch.cat(probs_T, dim=0)

    p_mean = probs_T.mean(dim=0)

    eps = 1e-8
    entropy = -torch.sum(p_mean * torch.log(p_mean + eps), dim=1)

    return entropy



def get_mcd_ood_scores(model, id_loader, ood_loader, T=20):
    model.to(device)

    id_scores = []
    ood_scores = []

    for x, _ in id_loader:
        x = x.to(device)
        entropy = mcd_entropy(model, x, T=T)

        id_scores.append(entropy.cpu().numpy())

    for x, _ in ood_loader:
        x = x.to(device)
        entropy = mcd_entropy(model, x, T=T)

        ood_scores.append(entropy.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores


In [7]:
net = CNN(dropout_p=0.3, num_classes=10)
train(net, train_id_loader, epochs=epochs, lr=lr)


Epoch 1; Train loss 1.7097687821863863; Accuracy 4713.04347826087
Epoch 2; Train loss 1.3547636185155805; Accuracy 6521.739130434783
Epoch 3; Train loss 1.2191441659732243; Accuracy 7192.838874680307
Epoch 4; Train loss 1.117754640176778; Accuracy 7704.859335038363
Epoch 5; Train loss 1.046035053480007; Accuracy 8047.314578005115
Epoch 6; Train loss 0.9835544826124635; Accuracy 8330.946291560102
Epoch 7; Train loss 0.949433871699721; Accuracy 8497.442455242966
Epoch 8; Train loss 0.9065384404433657; Accuracy 8686.18925831202
Epoch 9; Train loss 0.8750374440646842; Accuracy 8843.734015345268
Epoch 10; Train loss 0.8516317362065815; Accuracy 8961.125319693094


In [8]:
softmax_id_scores, softmax_ood_scores = get_softmax_ood_scores(net, test_id_loader, test_ood_loader)


In [9]:
metrics_auroc, metrics_aupr, metrics_fpr95 = compute_ood_metrics(softmax_id_scores, softmax_ood_scores)


AUROC 0.6867796850000001
AUPR 0.6190512130282325
FPR@95%TPR 0.7147


In [10]:
mcd_id_scores, mcd_ood_scores = get_mcd_ood_scores(net, test_id_loader, test_ood_loader)


In [11]:
mcd_auroc, mcd_aupr, mcd_fpr95 = compute_ood_metrics(mcd_id_scores, mcd_ood_scores)


AUROC 0.7560682350000001
AUPR 0.670794521172932
FPR@95%TPR 0.5938


# ReAct

In [12]:
def get_react_ood_scores(model, id_loader, ood_loader):
    model.to(device)

    id_scores = []
    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model(x, react=True)
            probs = F.softmax(logits, dim=1)
            top_score = probs.max(dim=1)[0]
            id_scores.append(top_score.cpu().numpy())
    
    ood_scores = []
    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model(x, react=True)
            probs = F.softmax(logits, dim=1)
            top_score = probs.max(dim=1)[0]
            ood_scores.append(top_score.cpu().numpy())
    
    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)
    return id_scores, ood_scores

    

In [13]:
react_id_scores, react_ood_scores = get_react_ood_scores(net, test_id_loader, test_ood_loader)


In [14]:
react_auroc, react_aupr, react_fpr95 = compute_ood_metrics(react_id_scores, react_ood_scores)


AUROC 0.35348607
AUPR 0.3931910255026625
FPR@95%TPR 0.9591
